# IMF extraction debugger — SB · Reform Targets · Prior Actions · QPC · TOC

Walks **every** extraction target through the real engine, stage by stage, and saves everything it sees to disk
so a run can be shared, diffed and replayed.

| Stage | What it shows | Needs API? |
|---|---|---|
| 0. Overview | date, report no., outline, TOC (Boxes/Annexes) with printed-vs-PDF page, Box captions the TOC missed | no |
| 1. Page detection | per-page regex verdicts (why a page was accepted / rejected), clusters, which cluster the engine kept, near-identical copies in discarded clusters | no |
| 2. Images | every detected page (chosen **and** discarded) rendered at the engine's 200 dpi and saved | no |
| 3. Claude calls | the *real* engine functions run with a tracing wrapper: exact image sent, prompt, every HTTP attempt, `stop_reason`, token usage, raw text, parsed JSON | yes (or replay/dry-run) |
| 4. Filtering | which raw rows were dropped by the engine and why | no |
| 5. Sanity checks | mixed Reform Area, SB↔PA overlap, truncated replies, header-as-title QPC, Boxes missing from TOC … | no |
| 6. Results + export | tables, engine-format JSON (feeds notebook 02), CSVs, HTML report, one ZIP | no |

**Modes** — set in the config cell: `DRY_RUN=True` (stages 0-2 only, no API calls, still saves every page image) ·
`REPLAY_DIR="…/<earlier run>"` (re-uses saved raw responses, so you can re-test parsing/filtering for free) ·
`PAGE_OVERRIDE` (force the pages for SB / Reform Target / PA / QPC when detection picks the wrong ones).

**Output layout** — `debug_output/<pdf name>/`: `sent/` (exact PNGs sent to Claude) · `candidates/` (detected pages, incl. discarded)
· `context/` · `raw/` (response text + full JSON) · `parsed/` · `prompts/` · `tables/` · `report.html` · `results_engine_format.json`
and `debug_output/<pdf name>_debug_<timestamp>.zip`.

> **Security:** this notebook never prints the API key. If an older notebook of yours printed it into a saved cell
> output, rotate the key and clear that output before sharing the file.


In [ ]:
# Install deps (skip any already satisfied)
%pip install -q pymupdf requests pandas matplotlib pillow python-dotenv openpyxl


## Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# ▶ CONFIG — the only cell you normally need to edit
# ════════════════════════════════════════════════════════════════════════════
PDF_PATHS = [
    "/Users/narminebenaissa/Downloads/pdf-section-extractor-main/IMF_Downloads_Renamed/LBR_DB/LBR_2008_EFF_Review4_Standalone.pdf",
]
PDF_INDEX = 0                    # which PDF in PDF_PATHS the stage-by-stage cells below work on

TARGETS = ["SB", "PA", "QPC"]    # which Claude-vision targets to run (SB also covers Reform Targets)

DRY_RUN     = None               # None = auto (dry run when no API key), True = never call the API, False = always call
REPLAY_DIR  = None               # e.g. "debug_output" (or one run folder): reuse the saved raw/*.txt of an earlier run instead of calling the API
SEQUENTIAL  = True               # one page at a time (rules out concurrency/rate-limit as a variable)

# Force pages when detection picks the wrong ones (PDF page numbers, 1-indexed). None = use the engine's detection.
PAGE_OVERRIDE = {"SB": None, "REFORM_TARGET": None, "PA": None, "QPC": None}

DEBUG_ROOT          = "debug_output"
DISPLAY_IMAGES      = True       # show page images inline
DISPLAY_MAX         = 12         # …but at most this many per stage (they are all saved regardless)
SAVE_CANDIDATE_PAGES = True      # also render pages that were detected but NOT chosen (discarded clusters)
CONTEXT_PAGES       = 1          # also save ±N neighbouring pages around every sent page (0 = off)
SAVE_ALL_PAGES      = False      # also save a low-res thumbnail of EVERY page (browse the whole document)
THUMB_DPI           = 50

ENGINE_PATH = None               # optional explicit path to imf_pdf_extraction_engine.py (else found via TOC_SCRIPT_DIR in .env)
ENGINE_MAX_TOKENS = 8192         # must match _call_claude_vision in the engine (the old debugger used 2048 -> could truncate)
ENGINE_DPI        = 200          # must match _render_page_and_call_claude in the engine


## Setup — load the engine
Finds `.env` and the engine file, prints version/capabilities and the run mode. The API key is never printed.

In [ ]:
import os, sys, glob, importlib.util
from pathlib import Path
from dotenv import load_dotenv, dotenv_values

# --- .env discovery (content-based, de-duplicated, never prints the key) -------
_ENV_NAMES = (".env", "env.example.txt", ".env.example.txt", "env.example", ".env.local")
_ENV_ROOTS = [
    Path.home() / "Downloads" / "pdf-section-extractor-main" / "Bilateral Surveillance Project",
    Path.cwd(),
    Path.home() / "Downloads" / "pdf-section-extractor-main",
]
_seen = []
for _r in _ENV_ROOTS:
    if _r.is_dir():
        for _n in _ENV_NAMES:
            _p = (_r / _n)
            if _p.is_file() and _p.resolve() not in _seen:
                _seen.append(_p.resolve())

def _usable(p):
    v = dotenv_values(p)
    return bool((v.get("ANTHROPIC_API_KEY") or "").strip().startswith("sk-ant-")
                or (v.get("TOC_SCRIPT_DIR") or v.get("TOC_EXTRACTOR_DIR") or "").strip())

ENV_PATH = next((p for p in _seen if _usable(p)), None)
if ENV_PATH:
    load_dotenv(ENV_PATH, override=True)
    print(f".env loaded from: {ENV_PATH}")
else:
    print("No usable .env found -- relying on environment variables / ENGINE_PATH.")

ANTHROPIC_API_KEY = (os.environ.get("ANTHROPIC_API_KEY") or "").strip()
HAVE_KEY = ANTHROPIC_API_KEY.startswith("sk-ant-")
TOC_SCRIPT_DIR = (os.environ.get("TOC_SCRIPT_DIR", "") or os.environ.get("TOC_EXTRACTOR_DIR", "")).strip()

# --- Engine discovery -------------------------------------------------------------
if ENGINE_PATH:
    _cands = [str(ENGINE_PATH)]
else:
    if not os.path.isdir(TOC_SCRIPT_DIR):
        raise RuntimeError(f"Set ENGINE_PATH in the config cell, or TOC_SCRIPT_DIR in .env (got {TOC_SCRIPT_DIR!r}).")
    _cands = [p for n in ("imf_pdf_extraction_engine.py", "toc_extractor.py")
              for p in glob.glob(os.path.join(os.path.abspath(TOC_SCRIPT_DIR), "**", n), recursive=True)]
if not _cands:
    raise FileNotFoundError(f"No engine file found under {TOC_SCRIPT_DIR!r}")
if len(_cands) > 1:
    print(f"NOTE: {len(_cands)} candidate engine files -- using the most recently modified:")
    for p in sorted(_cands, key=os.path.getmtime, reverse=True):
        print("   ", p)
ENGINE_FILE = max(_cands, key=os.path.getmtime)

for _m in list(sys.modules):
    if _m in ("imf_pdf_extraction_engine", "toc_extractor"):
        del sys.modules[_m]
_spec = importlib.util.spec_from_file_location("imf_pdf_extraction_engine", ENGINE_FILE)
te = importlib.util.module_from_spec(_spec)
sys.modules["imf_pdf_extraction_engine"] = te
_spec.loader.exec_module(te)
if hasattr(te, "set_api_key") and HAVE_KEY:
    te.set_api_key(ANTHROPIC_API_KEY)

CAPS = {
    "SB":  hasattr(te, "extract_structural_benchmarks"),
    "PA":  hasattr(te, "extract_prior_actions"),
    "QPC": hasattr(te, "extract_qpc"),
    "shared render wrapper": hasattr(te, "_render_page_and_call_claude"),
    "reform-target tables":  hasattr(te, "_find_benchmark_tables"),
}
print(f"\nEngine : {ENGINE_FILE}")
print(f"Version: {getattr(te, '__version__', 'MISSING')}   model: {getattr(te, '_VISION_MODEL', '?')}")
print("Capabilities:", ", ".join(f"{k}={'yes' if v else 'NO'}" for k, v in CAPS.items()))
print(f"API key: {'present' if HAVE_KEY else 'ABSENT'}")
_missing = [t for t in TARGETS if t in CAPS and not CAPS[t]]
if _missing:
    print(f"WARNING: this engine version has no extractor for {_missing}; those targets will be skipped.")

DRY = (not HAVE_KEY) if DRY_RUN is None else bool(DRY_RUN)
if REPLAY_DIR:
    print(f"MODE: REPLAY from {REPLAY_DIR} (no API calls)")
elif DRY:
    print("MODE: DRY RUN -- detection + images only, no API calls" + ("  (auto: no API key)" if DRY_RUN is None else ""))
else:
    print("MODE: LIVE -- Claude vision API calls will be made")


## Debug machinery (run once)
Tracing wrapper around the engine's API call, page-override patches, helpers. Nothing is called yet.

In [ ]:
import io, re, json, time, base64, zipfile, difflib, threading, traceback, contextlib, datetime as dt
import concurrent.futures as cf
import fitz, requests, pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, HTML, FileLink

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 220)
API_URL = "https://api.anthropic.com/v1/messages"

PROMPTS = {}
for _name, _attr in (("SB", "_VISION_PROMPT_SB_FULL"), ("PA", "_VISION_PROMPT_PRIOR_ACTIONS_FULL"),
                     ("QPC", "_VISION_PROMPT_QPC_FULL")):
    if hasattr(te, _attr):
        PROMPTS[getattr(te, _attr)] = _name

# ── run state ────────────────────────────────────────────────────────────────
class Run:
    """Everything about one PDF's debug run."""
    def __init__(self, pdf):
        self.pdf = str(pdf)
        self.stem = Path(pdf).stem
        self.dir = Path(DEBUG_ROOT) / self.stem
        for sub in ("sent", "candidates", "context", "raw", "parsed", "prompts", "tables") + (("pages_all",) if SAVE_ALL_PAGES else ()):
            (self.dir / sub).mkdir(parents=True, exist_ok=True)
        self.pages = {}            # effective pages per target
        self.kind_by_page = {}
        self.calls = []            # trace records for this run
        self.manifest = []         # every image written
        self.printed = {}          # pdf page -> printed page number
        self.sb = self.pa = self.qpc = None
        self.errors = {}
        self.t = {}

RUN = None
LOG_LOCK = threading.Lock()
_ctx = threading.local()
_ORIG_RENDER = getattr(te, "_render_page_and_call_claude", None)

def _add_manifest(R, path, role, page=None, target="", sent=False, img=None):
    im = img or Image.open(path)
    R.manifest.append(dict(path=str(Path(path).relative_to(R.dir)), role=role, pdf_page=page,
                           printed_page=R.printed.get(page), target=target, sent_to_claude=sent,
                           px=f"{im.size[0]}x{im.size[1]}", kb=round(Path(path).stat().st_size / 1024)))

# ── tracing replacement for te._call_claude_vision ───────────────────────────
def _traced_call(image_bytes, media_type, prompt, max_retries=4):
    R = RUN
    info = getattr(_ctx, "info", None)
    if info is None:
        info = _ctx.info = {}
    target = info.get("target") or PROMPTS.get(prompt, "OTHER")
    page = info.get("page")
    with LOG_LOCK:
        cid = len(R.calls) + 1
        rec = {"id": cid, "target": target, "page": page, "attempts": [], "dry_run": False, "replayed": False,
               "kind": R.kind_by_page.get(page, "") if target == "SB" else ""}
        R.calls.append(rec)
    info["rec"] = rec
    stem = f"{target}_p{page:03d}" if page else f"{target}_call{cid:03d}"
    rec["stem"] = stem

    # the exact bytes the engine is sending
    img_path = R.dir / "sent" / f"{stem}.png"
    img_path.write_bytes(image_bytes)
    im = Image.open(io.BytesIO(image_bytes))
    rec.update(image_path=str(img_path.relative_to(R.dir)), image_px=f"{im.size[0]}x{im.size[1]}",
               image_kb=round(len(image_bytes) / 1024))
    _add_manifest(R, img_path, "sent_to_claude", page, target, True, im)
    ppath = R.dir / "prompts" / f"{target}.txt"
    if not ppath.exists():
        ppath.write_text(prompt, encoding="utf-8")

    def _finish(text):
        rec["raw_chars"] = len(text)
        rec["raw_head"] = text[:80]
        (R.dir / "raw" / f"{stem}.txt").write_text(text, encoding="utf-8")
        return text

    if REPLAY_DIR:
        rp = Path(REPLAY_DIR) / "raw" / f"{stem}.txt"
        if not rp.is_file():                                  # REPLAY_DIR may also be the debug_output root
            rp = Path(REPLAY_DIR) / R.stem / "raw" / f"{stem}.txt"
        if rp.is_file():
            rec["replayed"] = True
            return _finish(rp.read_text(encoding="utf-8"))
        rec["error"] = f"no replay file {rp}"
        return ""
    if DRY:
        rec["dry_run"] = True
        return ""

    b64 = base64.standard_b64encode(image_bytes).decode("ascii")
    body = {"model": te._VISION_MODEL, "max_tokens": ENGINE_MAX_TOKENS,
            "messages": [{"role": "user", "content": [
                {"type": "image", "source": {"type": "base64", "media_type": media_type, "data": b64}},
                {"type": "text", "text": prompt}]}]}
    headers = {"content-type": "application/json", "x-api-key": ANTHROPIC_API_KEY,
               "anthropic-version": "2023-06-01"}
    for attempt in range(1, max_retries + 1):
        t0 = time.time()
        att = {"n": attempt}
        try:
            resp = requests.post(API_URL, json=body, headers=headers, timeout=90)
            att.update(status=resp.status_code, elapsed=round(time.time() - t0, 1))
            if resp.status_code == 200:
                j = resp.json()
                text = next((b["text"].strip() for b in j.get("content", []) if b.get("type") == "text"), "")
                u = j.get("usage", {}) or {}
                rec.update(stop_reason=j.get("stop_reason"), in_tokens=u.get("input_tokens"),
                           out_tokens=u.get("output_tokens"))
                rec["attempts"].append(att)
                (R.dir / "raw" / f"{stem}.response.json").write_text(json.dumps(j, indent=2, ensure_ascii=False),
                                                                     encoding="utf-8")
                return _finish(text)
            att["error"] = resp.text[:1500]
            rec["attempts"].append(att)
            if resp.status_code != 429 and resp.status_code < 500:
                break                                   # non-retryable (401/400/...)
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as exc:
            att.update(status="network", error=str(exc)[:500], elapsed=round(time.time() - t0, 1))
            rec["attempts"].append(att)
        except Exception as exc:
            att.update(status="exception", error=f"{type(exc).__name__}: {exc}"[:500])
            rec["attempts"].append(att)
            break
        if attempt < max_retries:
            time.sleep(min(2 ** attempt, 30))
    rec["error"] = (rec["attempts"][-1].get("error") if rec["attempts"] else "unknown")
    return ""

def _wrapped_render(pdf_path, page_num, prompt):
    """Tells the tracer which page/target a call is for, then runs the engine's own renderer + parser."""
    info = {"page": page_num, "target": PROMPTS.get(prompt, "OTHER")}
    _ctx.info = info
    parsed = _ORIG_RENDER(pdf_path, page_num, prompt)
    rec = info.get("rec")
    if rec is not None:
        rec["parsed"] = parsed
        rec["parsed_rows"] = (len(parsed) if isinstance(parsed, list)
                              else len(parsed.get("criteria") or []) if isinstance(parsed, dict) else 0)
        rec["parse_failed"] = bool(rec.get("raw_chars")) and parsed is None
        (RUN.dir / "parsed" / f"{rec['stem']}.json").write_text(
            json.dumps(parsed, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    return parsed

class _SerialExecutor:
    """Drop-in for ThreadPoolExecutor that runs jobs one at a time, in order."""
    def __init__(self, *a, **k): pass
    def __enter__(self): return self
    def __exit__(self, *a): return False
    def shutdown(self, *a, **k): pass
    def submit(self, fn, *a, **k):
        f = cf.Future()
        try:
            f.set_result(fn(*a, **k))
        except BaseException as exc:
            f.set_exception(exc)
        return f

@contextlib.contextmanager
def engine_patches():
    """Temporarily instrument the engine. Restored on exit, even on error."""
    saved = {}
    def patch(name, new):
        if hasattr(te, name):
            saved[name] = getattr(te, name)
            setattr(te, name, new)
    patch("_call_claude_vision", _traced_call)
    if _ORIG_RENDER is not None:
        patch("_render_page_and_call_claude", _wrapped_render)
    ov = PAGE_OVERRIDE or {}
    if ov.get("SB") or ov.get("REFORM_TARGET"):
        def _tables(pdf_path, include_reform_targets=True):
            out = []
            if ov.get("SB"):
                out.append({"kind": "SB", "pages": sorted(ov["SB"])})
            if ov.get("REFORM_TARGET") and include_reform_targets:
                out.append({"kind": "Reform Target", "pages": sorted(ov["REFORM_TARGET"])})
            return out
        patch("_find_benchmark_tables", _tables)
    if ov.get("PA"):
        patch("_find_prior_action_pages", lambda p: sorted(ov["PA"]))
    if ov.get("QPC"):
        patch("_find_qpc_pages", lambda p: sorted(ov["QPC"]))
    pool = cf.ThreadPoolExecutor
    if SEQUENTIAL:
        cf.ThreadPoolExecutor = _SerialExecutor
    try:
        yield
    finally:
        cf.ThreadPoolExecutor = pool
        for k, v in saved.items():
            setattr(te, k, v)

# ── small utilities ──────────────────────────────────────────────────────────
def render_png(doc, pg, dpi):
    return doc[pg - 1].get_pixmap(matrix=fitz.Matrix(dpi / 72, dpi / 72)).tobytes("png")

def printed_page_no(text):
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    for l in lines[:4] + lines[-4:]:
        m = re.fullmatch(r"(?:INTERNATIONAL MONETARY FUND\s+)?(\d{1,3})(?:\s+INTERNATIONAL MONETARY FUND)?", l, re.I)
        if m:
            return int(m.group(1))
    return None

def norm(s):
    return re.sub(r"\W+", " ", str(s or "").lower()).strip()

def show_png(png_bytes_or_path, title="", width=9):
    im = Image.open(io.BytesIO(png_bytes_or_path)) if isinstance(png_bytes_or_path, (bytes, bytearray)) else Image.open(png_bytes_or_path)
    h = width * im.size[1] / im.size[0]
    plt.figure(figsize=(width, min(h, 14)))
    plt.imshow(im); plt.axis("off"); plt.title(title, fontsize=10)
    plt.show()

def _table_caption(text):
    m = re.search(r"table\s+\d+[a-z]?[\.:][^\n]*", text, re.I)
    return m.group(0).strip()[:110] if m else ""

def _save_df(R, df, name):
    p = R.dir / "tables" / f"{name}.csv"
    df.to_csv(p, index=False, encoding="utf-8-sig")
    return p

print("helpers ready")


In [ ]:
def open_run(pdf):
    """Create the run folder, open the PDF once to map PDF page -> printed page number."""
    global RUN
    R = Run(pdf)
    RUN = R
    doc = fitz.open(R.pdf)
    R.n_pages = len(doc)
    R.printed = {i + 1: printed_page_no(doc[i].get_text()) for i in range(len(doc))}
    doc.close()
    print(f"▶ {R.stem}   ({R.n_pages} pages)   ->  {R.dir}")
    return R

# ── Stage 0: overview ────────────────────────────────────────────────────────
def stage_overview(R):
    doc = fitz.open(R.pdf)
    outline = doc.get_toc(simple=False)
    if hasattr(te, "_extract_document_date_from_doc"):
        R.date = te._extract_document_date_from_doc(doc, R.pdf)
    else:
        R.date = te.extract_document_date(R.pdf)
    R.is_si = te.is_selected_issues(R.pdf) if hasattr(te, "is_selected_issues") else False
    print(f"Document date : {R.date}")
    print(f"Pages         : {R.n_pages}   embedded outline entries: {len(outline)}   Selected-Issues paper: {R.is_si}")

    R.toc = [] if R.is_si else te.extract_toc_targets(R.pdf)
    src = "embedded outline (PDF page numbers)" if (outline and R.toc and
          any(t.category in ("Box", "Annex") for t in R.toc) and
          any(re.match(r"^\s*(BOX(?:ES)?|ANNEX(?:ES)?)\s*$", e[1], re.I) for e in outline)) else "text scan of Contents (PRINTED page numbers)"
    print(f"TOC source    : {src}")

    rows = []
    for t in R.toc:
        # locate the heading in the body to give the PDF page as well
        key = norm(re.sub(r"^[IVXLC\d]+\.\s*", "", t.title))[:35]
        found = None
        for i in range(min(10, R.n_pages), R.n_pages):
            for line in doc[i].get_text().splitlines():
                l = line.strip()
                if re.match(rf"^{t.category}\b", l, re.I) and key and key in norm(l):
                    found = i + 1
                    break
            if found:
                break
        rows.append(dict(category=t.category, title=t.title, toc_page=t.page, body_heading_on_pdf_page=found,
                         printed_on_that_page=R.printed.get(found)))
    R.toc_df = pd.DataFrame(rows)
    display(R.toc_df if len(R.toc_df) else "No Box/Annex entries found in the Contents.")

    # Box captions in the body that the TOC didn't list (e.g. boxes inside a bundled DSA)
    boxes = []
    for i in range(R.n_pages):
        for m in re.finditer(r"^\s*Box\s+(\d+[a-z]?)\.\s+(\S[^\n]{3,100})", doc[i].get_text(), re.M | re.I):
            title = m.group(2).strip()
            listed = any(t.category == "Box" and norm(title)[:30] in norm(t.title) for t in R.toc)
            boxes.append(dict(pdf_page=i + 1, printed=R.printed.get(i + 1), box=f"Box {m.group(1)}", title=title, in_toc=listed))
    R.boxes_df = pd.DataFrame(boxes)
    if len(R.boxes_df):
        print("\nBox captions found in the body:")
        display(R.boxes_df)
    else:
        print("\nNo 'Box N.' captions in the body.")
    doc.close()
    _save_df(R, R.toc_df, "toc")
    if len(R.boxes_df):
        _save_df(R, R.boxes_df, "box_captions")

# ── Stage 1: page detection ──────────────────────────────────────────────────
def _similarity(a, b):
    """max(sequence ratio on the first 3000 chars, word-set Jaccard) -- wide tables extract in a different text order,
    so the set overlap catches copies that the sequence ratio misses."""
    seq = difflib.SequenceMatcher(None, a[:3000], b[:3000], autojunk=False).ratio()
    ta, tb = set(a.split()), set(b.split())
    return max(seq, len(ta & tb) / max(1, len(ta | tb)))

def _sb_reject_reason(text):
    if not te._SB_TABLE_TITLE_RE.search(text):
        return ""
    if te._is_sb_table_page(text):
        return ""
    lines = [m.group(0) for m in re.finditer(r"table\s+\d+[a-z]?[\.:][^\n]*", text, re.I)
             if te._SB_TITLE_RE.search(m.group(0)) or re.search(r"reform\s+targets?", m.group(0), re.I)]
    if lines and all(te._is_prose_table_reference(l) for l in lines):
        return "sentence that mentions a table (prose reference)"
    return "narrative heading followed by numbered paragraphs (TMU-style)"

def stage_detect(R):
    doc = fitz.open(R.pdf)
    rows = []
    for i in range(R.n_pages):
        t = doc[i].get_text(); pg = i + 1
        sb_ok = te._is_sb_table_page(t)
        pa_ok = te._is_prior_action_table_page(t) if hasattr(te, "_is_prior_action_table_page") else False
        q_ok = te._is_qpc_table_page(t) if hasattr(te, "_is_qpc_table_page") else False
        rows.append(dict(
            page=pg, printed=R.printed.get(pg), table_no=te._extract_table_number(t),
            has_caption=bool(te._HAS_TABLE_CAPTION_RE.search(t)),
            mentions_SB=bool(te._SB_TEXT_RE.search(t)), SB=sb_ok,
            SB_kind=te._page_table_kind(t) if sb_ok else "", SB_rejected=_sb_reject_reason(t),
            mentions_PA=bool(re.search(r"prior\s+actions?", t, re.I)), PA=pa_ok,
            mentions_QPC=bool(re.search(r"performance\s+criteria", t, re.I)), QPC=q_ok,
            caption=_table_caption(t)))
    doc.close()
    R.scan = pd.DataFrame(rows)
    R.scan["printed"] = R.scan["printed"].astype("Int64")
    R.scan["table_no"] = R.scan["table_no"].astype(object)
    _save_df(R, R.scan, "page_scan")
    interesting = R.scan[R.scan[["mentions_SB", "SB", "PA", "QPC"]].any(axis=1) | (R.scan.SB_rejected != "")
                         | R.scan.mentions_PA & R.scan.has_caption]
    print("Pages with any SB / PA / QPC signal  (SB/PA/QPC = ACCEPTED by the engine's regex; SB_rejected = matched but filtered out):")
    display(interesting.drop(columns=["has_caption"]))

    # engine's own verdict (these log "Multiple captioned ... clusters" warnings -- that's expected)
    det_tables = te._find_benchmark_tables(R.pdf) if CAPS["reform-target tables"] else []
    det = {"SB": [], "Reform Target": [],
           "PA": te._find_prior_action_pages(R.pdf) if CAPS["PA"] else [],
           "QPC": te._find_qpc_pages(R.pdf) if CAPS["QPC"] else []}
    for t in det_tables:
        det[t["kind"]] = t["pages"]
    ov = PAGE_OVERRIDE or {}
    R.detected = det
    R.pages = {"SB": ov.get("SB") or det["SB"], "Reform Target": ov.get("REFORM_TARGET") or det["Reform Target"],
               "PA": ov.get("PA") or det["PA"], "QPC": ov.get("QPC") or det["QPC"]}
    R.kind_by_page = {p: k for k in ("SB", "Reform Target") for p in R.pages[k]}
    print("\nEngine-detected pages :", det)
    if any(ov.get(k) for k in ov):
        print("PAGE_OVERRIDE active  :", {k: v for k, v in ov.items() if v})
    print("Pages that will be used:", R.pages)

    # clusters: every cluster the engine formed, which one it kept, and whether discarded ones are copies
    cl_rows = []
    texts = {}
    d2 = fitz.open(R.pdf)
    def ptxt(pg):
        if pg not in texts:
            texts[pg] = norm(d2[pg - 1].get_text())
        return texts[pg]
    for target, flag, kept in (("SB/RT", "SB", det["SB"] + det["Reform Target"]), ("PA", "PA", det["PA"]), ("QPC", "QPC", det["QPC"])):
        pages = [int(r.page) for r in R.scan.itertuples() if getattr(r, flag)]
        if not pages:
            continue
        numbers = {int(r.page): (None if pd.isna(r.table_no) else r.table_no) for r in R.scan.itertuples() if getattr(r, flag)}
        for ci, c in enumerate(te._cluster_pages(pages, 5, numbers)):
            chosen = bool(set(c) & set(kept))
            sim = None if chosen or not kept else round(_similarity(" ".join(ptxt(p) for p in c), " ".join(ptxt(p) for p in kept)), 2)
            cl_rows.append(dict(target=target, cluster=ci, pages=c, printed=[R.printed.get(p) for p in c],
                                captioned=any(R.scan.set_index("page").loc[p, "has_caption"] for p in c),
                                chosen=chosen, similarity_to_chosen=sim,
                                caption=R.scan.set_index("page").loc[c[0], "caption"]))
    d2.close()
    R.clusters = pd.DataFrame(cl_rows)
    print("\nClusters (similarity_to_chosen ≈ 1.0 means a discarded cluster is just another printed copy of the chosen table):")
    display(R.clusters)
    _save_df(R, R.clusters, "clusters")

# ── Stage 2: render + save images ────────────────────────────────────────────
def stage_images(R):
    doc = fitz.open(R.pdf)
    saved = set()
    def save(pg, folder, label, role, dpi=ENGINE_DPI):
        p = R.dir / folder / f"p{pg:03d}_{label}.png"
        b = render_png(doc, pg, dpi)
        p.write_bytes(b)
        _add_manifest(R, p, role, pg, label)
        saved.add(p)
        return b

    # (a) discarded / alternative candidate pages
    if SAVE_CANDIDATE_PAGES and len(R.clusters):
        for r in R.clusters.itertuples():
            if not r.chosen:
                for pg in r.pages:
                    save(pg, "candidates", f"{r.target.replace('/', '-')}_cluster{r.cluster}_DISCARDED", "candidate_discarded")
    # (b) context pages around everything that will be sent
    if CONTEXT_PAGES:
        sent = sorted({p for v in R.pages.values() for p in v})
        ctx = sorted({q for p in sent for q in range(p - CONTEXT_PAGES, p + CONTEXT_PAGES + 1)
                      if 1 <= q <= R.n_pages and q not in sent})
        for pg in ctx:
            save(pg, "context", "context", "context")
    # (c) optional: every page as a low-res thumbnail
    if SAVE_ALL_PAGES:
        for pg in range(1, R.n_pages + 1):
            p = R.dir / "pages_all" / f"p{pg:03d}.png"
            p.write_bytes(render_png(doc, pg, THUMB_DPI))
            _add_manifest(R, p, "all_pages_thumb", pg)
    # (d) show what will be sent (the sent/ copies are written by the tracer in stage 3, byte-identical to the request)
    shown = 0
    for target in ("SB", "Reform Target", "PA", "QPC"):
        for pg in R.pages[target]:
            if DISPLAY_IMAGES and shown < DISPLAY_MAX:
                show_png(render_png(doc, pg, ENGINE_DPI), f"{target} — PDF page {pg} (printed {R.printed.get(pg)})")
                shown += 1
    doc.close()
    n_cand = sum(1 for m in R.manifest if m["role"] == "candidate_discarded")
    print(f"Saved {len(saved)} extra image(s) so far (discarded candidates: {n_cand}). "
          f"Images actually sent to Claude are saved in stage 3 -> {R.dir/'sent'}")

# ── Stage 3: run the real engine extractors under the tracer ─────────────────
def _timed(fn, *a):
    t0 = time.time()
    try:
        return fn(*a), None, round(time.time() - t0, 1)
    except Exception:
        return None, traceback.format_exc(), round(time.time() - t0, 1)

def stage_extract(R):
    R.calls.clear()
    jobs = []
    if "SB" in TARGETS and CAPS["SB"]:
        jobs.append(("SB", te.extract_structural_benchmarks))
    if "PA" in TARGETS and CAPS["PA"]:
        jobs.append(("PA", te.extract_prior_actions))
    if "QPC" in TARGETS and CAPS["QPC"]:
        jobs.append(("QPC", te.extract_qpc))
    with engine_patches():
        for name, fn in jobs:
            res, err, secs = _timed(fn, R.pdf)
            setattr(R, name.lower(), res)
            R.t[name] = secs
            if err:
                R.errors[name] = err
                print(f"✗ {name} raised after {secs}s:\n{err}")
            else:
                n = sum(len(x.rows) if hasattr(x, "rows") else len(x.criteria) for x in (res or []))
                print(f"✓ {name}: {len(res or [])} table(s), {n} row(s) in {secs}s")
    R.calls_df = pd.DataFrame([{
        "id": r["id"], "target": r["target"], "kind": r.get("kind", ""), "page": r["page"],
        "printed": R.printed.get(r["page"]), "mode": "replay" if r["replayed"] else "dry" if r["dry_run"] else "live",
        "attempts": len(r["attempts"]), "http": (r["attempts"][-1].get("status") if r["attempts"] else None),
        "secs": (r["attempts"][-1].get("elapsed") if r["attempts"] else None),
        "in_tok": r.get("in_tokens"), "out_tok": r.get("out_tokens"), "stop": r.get("stop_reason"),
        "raw_chars": r.get("raw_chars"), "parsed_rows": r.get("parsed_rows"), "parse_failed": r.get("parse_failed"),
        "image": r.get("image_px"), "img_kb": r.get("image_kb"), "error": (r.get("error") or "")[:120]}
        for r in R.calls])
    print("\nEvery Claude call made by the engine (one row per page image sent):")
    display(R.calls_df)
    if len(R.calls_df):
        _save_df(R, R.calls_df, "calls_log")
    (R.dir / "calls_trace.json").write_text(json.dumps(
        [{k: v for k, v in r.items() if k != "parsed"} for r in R.calls], indent=2, default=str), encoding="utf-8")

    shown = 0
    for r in R.calls:
        if DISPLAY_IMAGES and shown < DISPLAY_MAX:
            print("=" * 78)
            print(f"{r['target']} · PDF page {r['page']} · {r.get('image_px')} · stop_reason={r.get('stop_reason')} · "
                  f"{'replayed' if r['replayed'] else 'dry run' if r['dry_run'] else 'live'}")
            raw = (R.dir / "raw" / f"{r['stem']}.txt")
            if raw.exists():
                txt = raw.read_text(encoding="utf-8")
                print(f"RAW response ({len(txt)} chars):\n{txt[:1800]}{' …' if len(txt) > 1800 else ''}")
            elif r.get("error"):
                print("ERROR:", r["error"][:800])
            shown += 1

# ── Stage 4: what the engine dropped ─────────────────────────────────────────
_PRIOR_ACTION_RE = re.compile(      # copy of the backstop regex inside extract_structural_benchmarks
    r"^\s*Prior\s+Action\s*:|submit\s+to\s+parliament\s+\d{4}\s+budget"
    r"|submit\s+to\s+parliament.*budget\s+in\s+line\s+with\s+program", re.I)

def stage_dropped(R):
    final_sb = {k for t in (R.sb or []) for k in t.rows}
    final_pa = {k for t in (R.pa or []) for k in t.rows}
    final_q = {c.criterion for t in (R.qpc or []) for c in t.criteria}
    out, raw_rows = [], []
    for r in R.calls:
        parsed = r.get("parsed")
        if r["target"] in ("SB", "PA") and isinstance(parsed, list):
            for it in parsed:
                if not isinstance(it, dict):
                    out.append(dict(target=r["target"], page=r["page"], reason="item is not an object", text=str(it)[:120]))
                    continue
                raw_rows.append(dict(target=r["target"], page=r["page"], **{k: it.get(k) for k in
                                 ("reform_area", "benchmark", "action", "due_date", "status", "category")}))
                if r["target"] == "SB":
                    b = (it.get("benchmark") or "").strip()
                    why = ("empty benchmark" if not b else "shorter than 15 chars" if len(b) < 15 else
                           "Prior-Action backstop regex" if _PRIOR_ACTION_RE.search(b) else
                           "" if b in final_sb else "merged / overwritten (duplicate text or concluded page)")
                else:
                    b = (it.get("action") or "").strip()
                    why = ("empty action" if not b else "shorter than 10 chars" if len(b) < 10 else
                           "" if b in final_pa else "merged / overwritten (duplicate text)")
                if why:
                    out.append(dict(target=r["target"], page=r["page"], reason=why, text=b[:120]))
        elif r["target"] == "QPC" and isinstance(parsed, dict):
            for c in parsed.get("criteria") or []:
                raw_rows.append(dict(target="QPC", page=r["page"], category=c.get("category"), benchmark=c.get("criterion")))
                if not (c.get("criterion") or "").strip():
                    out.append(dict(target="QPC", page=r["page"], reason="empty criterion", text=str(c)[:120]))
    R.raw_rows_df = pd.DataFrame(raw_rows)
    R.dropped_df = pd.DataFrame(out)
    if len(R.raw_rows_df):
        _save_df(R, R.raw_rows_df, "raw_rows_from_claude")
    print(f"Raw rows returned by Claude: {len(R.raw_rows_df)}   dropped/merged by the engine: {len(R.dropped_df)}")
    display(R.dropped_df if len(R.dropped_df) else "Nothing dropped.")
    if len(R.dropped_df):
        _save_df(R, R.dropped_df, "dropped_rows")

# ── Stage 5: sanity checks ───────────────────────────────────────────────────
def stage_sanity(R):
    out = []
    add = lambda sev, check, detail: out.append(dict(severity=sev, check=check, detail=detail))
    if R.calls and all(r["dry_run"] for r in R.calls):
        add("INFO", "dry_run", "No API calls were made -- extraction checks are skipped; images, detection and prompts were still saved")
    for r in R.calls:
        lab = f"{r['target']} p{r['page']}"
        if r.get("error"):
            add("ERROR", "api_failed", f"{lab}: {str(r['error'])[:160]}")
        if r.get("stop_reason") == "max_tokens":
            add("ERROR", "truncated_reply", f"{lab}: hit max_tokens ({r.get('out_tokens')}) -> JSON cut off, rows lost")
        if r.get("parse_failed"):
            add("ERROR", "json_parse_failed", f"{lab}: reply could not be parsed ({r.get('raw_chars')} chars)")
        if not r["dry_run"] and not r.get("error") and not r.get("parse_failed") and r.get("parsed_rows") == 0:
            add("WARN", "zero_rows", f"{lab}: valid reply but 0 rows (page may not hold that table)")
        if r.get("raw_head") and not r["raw_head"].lstrip().startswith(("[", "{")) and not r.get("parse_failed"):
            add("INFO", "preamble_before_json", f"{lab}: reply starts with prose; parser fell back to first bracket")
        w, h = (int(x) for x in r["image_px"].split("x"))
        if max(w, h) > 8000 or r.get("image_kb", 0) > 5000:
            add("WARN", "image_too_large", f"{lab}: {r['image_px']} / {r['image_kb']} KB (API limits ~8000 px / 5 MB)")
    # row-level checks
    sb_rows = [(t, k, v) for t in (R.sb or []) for k, v in t.rows.items()]
    for kind in ("SB", "Reform Target"):
        sub = [(t, k, v) for t, k, v in sb_rows if v.get("Type") == kind]
        if sub:
            ra = [v.get("Reform Area") for _, _, v in sub]
            filled = sum(1 for x in ra if x)
            if 0 < filled < len(ra):
                add("WARN", "reform_area_mixed", f"{kind}: {filled}/{len(ra)} rows have a Reform Area, the rest are null "
                    f"(check whether the table really has that column or only an 'Objective')")
            if len({x for x in ra if x}) == len([x for x in ra if x]) and filled > 3:
                add("INFO", "reform_area_all_distinct", f"{kind}: every Reform Area value is different -> probably free-text Objective, not a category")
            longs = [v.get("Status") for _, _, v in sub if v.get("Status") and len(v["Status"]) > 25]
            if longs:
                add("INFO", "long_status", f"{kind}: {len(longs)} status value(s) > 25 chars, e.g. {longs[0]!r}")
            nostat = sum(1 for _, _, v in sub if not v.get("Status"))
            if nostat:
                add("INFO", "blank_status", f"{kind}: {nostat}/{len(sub)} rows have no status (notebook 02 resolves these to 'New' or blank)")
            nodate = sum(1 for _, _, v in sub if not v.get("Due date"))
            if nodate:
                add("WARN", "blank_due_date", f"{kind}: {nodate}/{len(sub)} rows have no due date")
    # SB <-> PA overlap
    if R.pa and sb_rows:
        for pt in R.pa:
            for act in pt.rows:
                for _, k, v in sb_rows:
                    if difflib.SequenceMatcher(None, norm(act)[:100], norm(k)[:100]).ratio() > 0.85:
                        add("WARN", "same_action_in_PA_and_SB", f"{act[:70]!r} appears in both (SB status {v.get('Status')!r}) -> double-counted downstream")
    # QPC
    for q in (R.qpc or []):
        tt = q.table_title or ""
        if len(tt) < 20 and tt.upper() == tt or "table" not in tt.lower():
            add("WARN", "qpc_title_not_a_caption", f"QPC p{q.page_num}: title is {tt!r} (page header, not the 'Table N.' caption)")
        if not q.unit_note:
            add("INFO", "qpc_no_unit_note", f"QPC p{q.page_num}: no unit note extracted")
        shapes = {tuple(sorted((td.get('values') or {}).keys())) for c in q.criteria for td in c.test_dates if isinstance(td, dict)}
        if len(shapes) > 1:
            add("INFO", "qpc_mixed_subcolumns", f"QPC p{q.page_num}: {len(shapes)} different sub-column sets across test dates (normal for past vs future dates)")
        if any(not c.test_dates for c in q.criteria):
            add("WARN", "qpc_criterion_without_dates", f"QPC p{q.page_num}: {sum(1 for c in q.criteria if not c.test_dates)} criteria have no test dates")
        add("INFO", "qpc_test_date_type_not_stored", "PC / IT tag printed under each test date has no field in the schema")
    # detection-level
    if len(R.clusters):
        for tgt, g in R.clusters.groupby("target"):
            cap = g[g.captioned]
            if len(cap) > 1 and (~cap.chosen).any():
                sims = [s for s in cap[~cap.chosen].similarity_to_chosen if s is not None]
                add("INFO", "multiple_captioned_clusters",
                    f"{tgt}: {len(cap)} captioned clusters, engine keeps the LAST one; discarded similarity to kept = {sims} "
                    f"({'looks like a duplicate printed copy' if sims and min(sims) > 0.8 else 'DIFFERENT content - verify the right table was kept'})")
    for tgt in ("SB", "PA", "QPC"):
        if tgt in TARGETS and not (R.pages.get(tgt) or (tgt == "SB" and R.pages.get("Reform Target"))):
            add("WARN", "no_pages_detected", f"{tgt}: no pages detected -> nothing sent to Claude")
    if len(getattr(R, "boxes_df", [])) and not R.boxes_df.in_toc.all():
        miss = R.boxes_df[~R.boxes_df.in_toc]
        add("WARN", "boxes_missing_from_toc", f"{len(miss)} Box caption(s) in the body are not in the Contents, e.g. {miss.iloc[0].box} on PDF p.{miss.iloc[0].pdf_page}")
    for tgt, err in R.errors.items():
        add("ERROR", "extractor_raised", f"{tgt}: {err.strip().splitlines()[-1]}")
    R.sanity_df = pd.DataFrame(out, columns=["severity", "check", "detail"])
    order = {"ERROR": 0, "WARN": 1, "INFO": 2}
    R.sanity_df = R.sanity_df.sort_values("severity", key=lambda s: s.map(order)).reset_index(drop=True)
    print(R.sanity_df.severity.value_counts().to_dict())
    display(R.sanity_df)
    _save_df(R, R.sanity_df, "sanity_checks")

# ── Stage 6: results + export ────────────────────────────────────────────────
def stage_results(R):
    R.sb_df = pd.DataFrame([dict(type=v.get("Type"), page=t.page_num, printed=R.printed.get(t.page_num),
                                 reform_area=v.get("Reform Area"), benchmark=k, due_date=v.get("Due date"),
                                 status=v.get("Status"), table=t.table_title)
                            for t in (R.sb or []) for k, v in t.rows.items()])
    R.pa_df = pd.DataFrame([dict(page=t.page_num, printed=R.printed.get(t.page_num), category=v.get("Category"),
                                 reform_area=v.get("Reform Area"), action=k, due_date=v.get("Due date"),
                                 status=v.get("Status"), table=t.table_title)
                            for t in (R.pa or []) for k, v in t.rows.items()])
    qlong = [dict(page=q.page_num, printed=R.printed.get(q.page_num), table=q.table_title, unit=q.unit_note,
                  category=c.category, criterion=c.criterion, test_date=td.get("date"), sub=sub, value=val)
             for q in (R.qpc or []) for c in q.criteria for td in c.test_dates if isinstance(td, dict)
             for sub, val in (td.get("values") or {}).items()]
    R.qpc_long_df = pd.DataFrame(qlong)
    R.qpc_wide_df = (R.qpc_long_df.assign(col=lambda d: d.test_date.astype(str) + " — " + d["sub"])
                     .pivot_table(index=["category", "criterion"], columns="col", values="value", aggfunc="first", sort=False)
                     .reset_index() if len(R.qpc_long_df) else pd.DataFrame())
    for name, df in (("SB / Reform Targets", R.sb_df), ("Prior Actions", R.pa_df), ("QPC (wide)", R.qpc_wide_df)):
        print(f"\n=== {name}: {len(df)} row(s) ===")
        display(df if len(df) else "— none —")
    for name, df in (("sb_rows", R.sb_df), ("pa_rows", R.pa_df), ("qpc_long", R.qpc_long_df), ("qpc_wide", R.qpc_wide_df)):
        if len(df):
            _save_df(R, df, name)

    # engine-format JSON: same shape as extract_all_targets(), so notebook 02 can consume it
    engine_json = {"document_date": R.date, "si_title": None,
                   "toc": [t.to_dict() for t in R.toc], "benchmarks": [b.to_dict() for b in (R.sb or [])],
                   "prior_actions": [p.to_dict() for p in (R.pa or [])], "qpc": [q.to_dict() for q in (R.qpc or [])]}
    (R.dir / "results_engine_format.json").write_text(json.dumps(engine_json, indent=2, ensure_ascii=False), encoding="utf-8")

    # HTML report: each sent page next to the rows that came out of it
    parts = [f"<html><head><meta charset='utf-8'><style>body{{font-family:sans-serif;margin:24px}}"
             f"table{{border-collapse:collapse;font-size:12px}}td,th{{border:1px solid #bbb;padding:3px 6px;vertical-align:top}}"
             f".row{{display:flex;gap:20px;margin-bottom:36px}}img{{max-width:560px;border:1px solid #888}}</style></head><body>",
             f"<h1>{R.stem}</h1><p>{R.n_pages} pages · date {R.date.get('raw')} · report {R.date.get('report_number')} · "
             f"engine {getattr(te, '__version__', '?')} · mode {'replay' if REPLAY_DIR else 'dry' if DRY else 'live'}</p>"]
    if len(R.sanity_df):
        parts.append("<h2>Sanity checks</h2>" + R.sanity_df.to_html(index=False, escape=True))
    for r in sorted(R.calls, key=lambda x: (x["page"] or 0, x["target"])):
        parsed = r.get("parsed")
        if r["target"] == "QPC" and isinstance(parsed, dict):
            body = pd.DataFrame([{"category": c.get("category"), "criterion": c.get("criterion"),
                                  "dates": len(c.get("test_dates") or [])} for c in parsed.get("criteria") or []]).to_html(index=False)
        elif isinstance(parsed, list) and parsed:
            body = pd.DataFrame(parsed).to_html(index=False, escape=True)
        else:
            body = "<i>no rows (dry run, failure or empty)</i>"
        parts.append(f"<h2>{r['target']} · PDF p.{r['page']} (printed {R.printed.get(r['page'])})</h2>"
                     f"<div class='row'><a href='{r['image_path']}'><img src='{r['image_path']}'></a><div>{body}</div></div>")
    parts.append("</body></html>")
    (R.dir / "report.html").write_text("\n".join(parts), encoding="utf-8")

    pd.DataFrame(R.manifest).to_csv(R.dir / "image_manifest.csv", index=False)
    ts = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    zpath = Path(DEBUG_ROOT) / f"{R.stem}_debug_{ts}.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(R.dir.rglob("*")):
            if f.is_file():
                z.write(f, f.relative_to(Path(DEBUG_ROOT)))
    R.zip = zpath
    man = pd.DataFrame(R.manifest)
    print(f"\nSaved to {R.dir}")
    if len(man):
        print("Images saved:", man.groupby("role").size().to_dict(), f"· total {len(man)} · {man.kb.sum()/1024:.1f} MB")
    print(f"Report : {R.dir/'report.html'}")
    print(f"ZIP    : {zpath}  ({zpath.stat().st_size/1024/1024:.1f} MB)")
    display(FileLink(str(zpath)))

def run_all(pdf):
    """Run every stage for one PDF (used by the batch cell)."""
    R = open_run(pdf)
    for fn in (stage_overview, stage_detect, stage_images, stage_extract, stage_dropped, stage_sanity, stage_results):
        print("\n" + "─" * 30, fn.__name__, "─" * 30)
        fn(R)
    return R

# one-page experiment: re-run a single page with the engine prompt or your own
def probe(pdf, page, target="SB", prompt=None):
    """Send ONE page and print the raw reply. target in SB/PA/QPC picks the engine prompt; pass prompt= to try your own."""
    global RUN
    if RUN is None or RUN.pdf != str(pdf):
        RUN = Run(pdf)
    name2prompt = {v: k for k, v in PROMPTS.items()}
    p = prompt or name2prompt[target]
    with engine_patches():
        parsed = te._render_page_and_call_claude(str(pdf), page, p)
    rec = RUN.calls[-1]
    print(f"stop_reason={rec.get('stop_reason')}  tokens in/out={rec.get('in_tokens')}/{rec.get('out_tokens')}  error={rec.get('error')}")
    print((RUN.dir / "raw" / f"{rec['stem']}.txt").read_text(encoding="utf-8")[:3000] if (RUN.dir / "raw" / f"{rec['stem']}.txt").exists() else "(no raw text)")
    return parsed

print("stage functions ready")


## Open the run for `PDF_PATHS[PDF_INDEX]`

In [ ]:
R = open_run(PDF_PATHS[PDF_INDEX])

## 0. Overview — date, report number, TOC (Boxes / Annexes)
`body_heading_on_pdf_page` vs `toc_page` shows whether the TOC page numbers are *printed* numbers or PDF indices. Box captions in the body that are not in the Contents (e.g. a box inside a bundled DSA) are listed below it.

In [ ]:
stage_overview(R)

## 1. Page detection (regex / fitz — no Claude)
Per-page verdicts from the engine's own predicates. `SB_rejected` explains pages that matched the title regex but were filtered (prose reference, TMU narrative heading). The cluster table shows what was kept **and** what was discarded; `similarity_to_chosen ≈ 1` means the discarded cluster is a second printed copy of the same table (e.g. staff-report table vs MEFP table).

In [ ]:
stage_detect(R)

## 2. Images — what Claude will see
Renders every detected page at the engine's 200 dpi and **saves** discarded candidates, context pages (and optionally every page). The exact bytes sent to Claude are saved by the tracer in stage 3 under `sent/`.

In [ ]:
stage_images(R)

## 3. Claude calls — the real engine under a tracer
Runs `extract_structural_benchmarks` (SB **and** Reform Targets), `extract_prior_actions` and `extract_qpc` exactly as the pipeline does, with the API call wrapped: saves the image sent, prompt, every HTTP attempt, `stop_reason` (`max_tokens` = truncated JSON), token usage, raw text and parsed JSON. Sequential by default.

In [ ]:
stage_extract(R)

## 4. What the engine dropped
Raw rows from Claude vs rows that survived the engine's filters (length cut-offs, Prior-Action backstop, duplicate-key overwrite, “(Concluded)” merge).

In [ ]:
stage_dropped(R)

## 5. Sanity checks
Problems seen in real runs: truncated replies, mixed/null Reform Area, the same action listed as both PA and SB, QPC title taken from the page header, Boxes missing from the Contents, multiple captioned clusters.

In [ ]:
stage_sanity(R)

## 6. Results + export
Tables, `results_engine_format.json` (same shape as `extract_all_targets()` → can go straight into notebook 02), CSVs, `report.html` (page image next to the rows extracted from it) and one ZIP of everything.

In [ ]:
stage_results(R)

## Batch — run every stage for the remaining PDFs in `PDF_PATHS`

In [ ]:
for _p in PDF_PATHS[PDF_INDEX + 1:]:
    run_all(_p)

## Probe one page
Re-send a single page with the engine prompt (or your own via `prompt=`) and read the raw reply — useful for prompt tuning.
```python
parsed = probe(PDF_PATHS[0], 115, target="SB")
```

## Notes
- **Wrong pages?** Stage 1 tells you why (rejected reason, clusters). Force pages with `PAGE_OVERRIDE` and re-run stages 2-6.
- **Right pages, wrong content?** Stage 3 raw text + saved image: is the table in the image? Was the reply truncated (`stop_reason`)? Then check stage 4 (dropped) and 5 (sanity). The fix is usually in the prompt, not a regex.
- **Iterating on parsing/filters without paying for API calls:** set `REPLAY_DIR` to an earlier run folder; the saved `raw/*.txt` are fed back through the engine.
- Everything needed to reproduce a run is in the ZIP: the exact images, prompts, raw replies and tables.